In [1]:
import numpy as np
import pandas as pd
import scipy.ndimage as ndi
import skimage as ski
import glob
import cellpose.models as models
import nd2
import napari
import plotly.express as px



Welcome to CellposeSAM, cellpose v4.0.1! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 




In [2]:
model = models.CellposeModel(gpu=True)

In [3]:
img = ski.io.imread('Data/projections/5_animal1_oral_MAX.tif')

In [4]:
viewer = napari.Viewer()

In [5]:
def process_fname(fname, viewer=None, display=False):
    img = ski.io.imread(fname)
    rslts = model.eval(img, cellprob_threshold=-0.0, flow_threshold=0.7)
    labels = rslts[0]

    LoG = -ndi.gaussian_laplace(img.astype(np.float32), sigma=3)
    df = pd.DataFrame(ski.measure.regionprops_table(labels, LoG, properties=['centroid', 'mean_intensity', 'area', 'label']))
    LoG_img = ski.util.map_array(labels, df['label'].values, df['mean_intensity'].values)
    df['good'] = df['mean_intensity'] > 2.0
    filtered_labels = ski.util.map_array(labels, df['label'].values, (df['good']*df['label']).values)
    
    df['fname'] = fname
    df = df[df['good']]

    # Merging with Ruohan's annotation
    ring = ski.io.imread(fname.replace('.tif', '_ring.tiff'))
    mouth = ndi.binary_fill_holes(ring > 0)
    mouth = mouth & ~(ring > 0)
    blurred = ski.filters.gaussian(img.astype(np.float32), sigma=10)
    threshed = blurred > 120
    threshed = ndi.binary_fill_holes(threshed)
    edt = ndi.distance_transform_edt(threshed)
    animal = edt > 10
    animal = animal & ~(ring > 0) & ~(mouth > 0)
    master = ring + mouth*2 +animal*4

    df = pd.DataFrame(ski.measure.regionprops_table(filtered_labels, properties=['centroid', 'area', 'label']))
    df['state'] = master[df['centroid-0'].values.astype(int), df['centroid-1'].values.astype(int)]
    df['state'] = df['state'].map({0: 'background', 1: 'ring', 2: 'mouth', 4: 'tentacles'})
    df['fname'] = fname
    df['tentacle_area'] = np.sum((master == 4).astype(int))
    df['ring_area'] = np.sum((master == 1).astype(int))
    df['mouth_area'] = np.sum((master == 2).astype(int))


    if display:
        viewer.add_image(img)
        viewer.add_labels(labels, visible=False)
        viewer.layers[-1].contour = 1
        viewer.add_image(LoG_img, blending='additive', colormap='magenta', visible=False)
        viewer.add_labels(filtered_labels)
        viewer.add_labels(master, name='master')
    
    return df

In [8]:
fnames = glob.glob('Data/projections/*.tif')

In [9]:
df = []
for fname in fnames:
    viewer = napari.Viewer()
    df.append(process_fname(fname, viewer, display=True))
    viewer.screenshot(fname.replace('.tif', '_napari.png'))
    viewer.layers[-1].visible = False
    viewer.screenshot(fname.replace('.tif', '_napari_cellpose.png'))
df = pd.concat(df)

In [10]:
dy = 0.39
dx = 0.27

df['area'] = df['area'] * dy * dx
df['tentacle_area'] = df['tentacle_area'] * dy * dx
df['ring_area'] = df['ring_area'] * dy * dx
df['mouth_area'] = df['mouth_area'] * dy * dx

In [11]:
df.to_csv('AreaResults.csv')

In [12]:
px.box(df, x='state', y='area', points='all', animation_frame='fname', width=600, title='Cell area distribution across states and files')

In [13]:
agged = df.groupby(['fname', 'state'])['area'].median().reset_index()

In [14]:
f = px.box(agged, x='state', y='area', points='all', width=400, title='Median cell area distribution across states and files')
f.write_html('MedianAreaDistribution.html')
f

In [15]:
agged.to_csv('Aggregated.csv')

In [16]:
df['log_area'] = np.log(df['area'])
f = px.histogram(df, x='log_area', animation_frame='state', histnorm='percent', width=400, range_y=[0,10])
f.write_html('LogAreaDistribution.html')
f

# Fractional Areas by State

In [17]:
agged = df.groupby(['fname', 'state']).agg({'label':'count', 'area':'sum', 'tentacle_area':'median', 'ring_area':'median', 
                                            'mouth_area':'median'}).reset_index().rename(columns={'label':'cell_count'})

tagged = agged.melt(id_vars=['fname'], value_vars=['tentacle_area', 'ring_area', 'mouth_area'], var_name='region', value_name='region_area').drop_duplicates()
tagged['state'] = tagged['region'].map({'tentacle_area': 'tentacles', 'ring_area': 'ring', 'mouth_area': 'mouth'})

agged = agged.iloc[:,0:4].merge(tagged, on=['fname', 'state'])
agged['count_density'] = agged['cell_count'] / agged['region_area']
agged['area_density'] = agged['area'] / agged['region_area']
agged

,fname,state,cell_count,area,region,region_area,count_density,area_density
0,Data/projections\10_8ten4_561_oral_MAX.tif,mouth,36,422.3583,mouth_area,19294.8561,0.001866,0.021890
1,Data/projections\10_8ten4_561_oral_MAX.tif,ring,138,3263.6682,ring_area,19365.9336,0.007126,0.168526
2,Data/projections\10_8ten4_561_oral_MAX.tif,tentacles,951,15936.9444,tentacle_area,133689.6171,0.007113,0.119209
3,Data/projections\1_juvenile1_oral_Max.tif,mouth,118,1595.2950,mouth_area,54144.1017,0.002179,0.029464
4,Data/projections\1_juvenile1_oral_Max.tif,ring,238,7571.3859,ring_area,49951.2663,0.004765,0.151575
5,Data/projections\1_juvenile1_oral_Max.tif,tentacles,868,15316.2009,tentacle_area,253588.3038,0.003423,0.060398
6,Data/projections\2_juvenile5_oral001_Max.tif,mouth,70,939.0654,mouth_area,23895.0972,0.002929,0.039300
7,Data/projections\2_juvenile5_oral001_Max.tif,ring,219,7518.7359,ring_area,46246.0752,0.004736,0.162581
8,Data/projections\2_juvenile5_oral001_Max.tif,tentacles,1139,18274.1832,tentacle_area,216373.1778,0.005264,0.084457
9,Data/projections\3_juvenile6_oral001_Max.tif,mouth,89,1151.7714,mouth_area,42293.5344,0.002104,0.027233


In [18]:
agged.to_csv('DensityDistribution.csv')

In [19]:
f = px.box(agged, x='region', y='count_density', points='all', width=400, title='Cell Body Counts Per Unit Area of Region')
f.write_html('CountDensityDistribution.html')
f

In [22]:
agged

,fname,state,cell_count,area,region,region_area,count_density,area_density
0,Data/projections\10_8ten4_561_oral_MAX.tif,mouth,36,422.3583,mouth_area,19294.8561,0.001866,0.021890
1,Data/projections\10_8ten4_561_oral_MAX.tif,ring,138,3263.6682,ring_area,19365.9336,0.007126,0.168526
2,Data/projections\10_8ten4_561_oral_MAX.tif,tentacles,951,15936.9444,tentacle_area,133689.6171,0.007113,0.119209
3,Data/projections\1_juvenile1_oral_Max.tif,mouth,118,1595.2950,mouth_area,54144.1017,0.002179,0.029464
4,Data/projections\1_juvenile1_oral_Max.tif,ring,238,7571.3859,ring_area,49951.2663,0.004765,0.151575
5,Data/projections\1_juvenile1_oral_Max.tif,tentacles,868,15316.2009,tentacle_area,253588.3038,0.003423,0.060398
6,Data/projections\2_juvenile5_oral001_Max.tif,mouth,70,939.0654,mouth_area,23895.0972,0.002929,0.039300
7,Data/projections\2_juvenile5_oral001_Max.tif,ring,219,7518.7359,ring_area,46246.0752,0.004736,0.162581
8,Data/projections\2_juvenile5_oral001_Max.tif,tentacles,1139,18274.1832,tentacle_area,216373.1778,0.005264,0.084457
9,Data/projections\3_juvenile6_oral001_Max.tif,mouth,89,1151.7714,mouth_area,42293.5344,0.002104,0.027233


In [23]:
f = px.box(agged, x='region', y='area_density', points='all', width=400, title='Cell Body Area Per Unit Area of Region', hover_data=['fname'])
f.write_html('AreaDensityDistribution.html')
f